# Notebook 05 — AI-written legitimate false alarms + LLM rewriting robustness (E01)

**CPU-first / Kaggle Run All / research benchmark, not a claim of verified human annotations.**

Question: Does a phishing classifier trained on source-documented original messages mistakenly flag legitimate emails rewritten by GPT-4o/DeepSeek more often than legitimate originals from the **same held-out parent IDs**? All LLM rewrite rows are **derived**; they are not independent external datasets. Original `Valid` and `Phishing` are **source-assigned** classes, not independently adjudicated ground truth.

Safety and science: `Spam`, `Phishing Simulation` and conflicting/unknown classes excluded. The D2 original/rewrites are joined on `No.`, and **no parent may cross train/validation/test**. Exact normalized text duplication between train and held-out groups is quarantined. No test-derived threshold fitting, public raw text, or per-record outputs. D2/D3 documented common lineage prohibits use of D3 as an independent test. Optional D4 is deliberately excluded from this first matched-parent experiment owing to source/generator and language confounds.

**Output:** aggregate JSON, CSV metrics, human-readable summary, run manifest under `/kaggle/working/phishing_n05_safe/`. Raw source and per-row data remain in `/kaggle/temp/phishing_n05_private/` or verified private Kaggle Input. **No GPU required.** Internet can be disabled if exact pinned source files already exist in a private Kaggle Input.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, precision_score,
    recall_score, average_precision_score, roc_auc_score, brier_score_loss, confusion_matrix)
from sklearn.calibration import CalibratedClassifierCV
import csv, hashlib, json, os, platform, re, shutil, sys, tempfile, time, urllib.parse, urllib.request
import numpy as np

RELEASE='N05-E01-v1.0-2026-11-02'
D2_REV='d13e9bb73d03f458e6d5b6cb6c7b0a708ea47e3d'
SEED=2026
ROOT=Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd()
PRIV=(Path('/kaggle/temp') if Path('/kaggle').exists() else Path(tempfile.gettempdir()))/'phishing_n05_private'
OUT=ROOT/'phishing_n05_safe'; OUT.mkdir(parents=True,exist_ok=True); PRIV.mkdir(parents=True,exist_ok=True)
SOURCE={
 'original':('1_DataSet/cleaned_emails.json',7370034,'b9d6dd2ac3d6e2d337b9b9d79fc8f1f4377ea4f9'),
 'deepseek':('3_RePhrase/DeepSeek/emails_rephrased_clean_deepseek.json',7528880,'11cc02aeb03b5a318f3107a71a20f2406b8f790a'),
 'gpt4o':('3_RePhrase/GPT4/emails_rephrased_clean_openai4o.json',7079916,'d0cf5fe3bdf0a3dd962492720c3cb10381f9e313'),
 'randomapi':('3_RePhrase/RandomAPI/emails_rephrased_openrouter_clean.json',7124691,'2febaf0e74ac23689889164bfe5b8e538b706140')
}

def git_blob(p):
    h=hashlib.sha1(b'blob '+str(p.stat().st_size).encode()+b'\x00')
    with p.open('rb') as f:
        for b in iter(lambda:f.read(1<<20),b''):h.update(b)
    return h.hexdigest()

def sha256(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for b in iter(lambda:f.read(1<<20),b''):h.update(b)
    return h.hexdigest()

def acquire(name):
    rel,size,blob=SOURCE[name]; filename=Path(rel).name
    candidates=[]
    if Path('/kaggle/input').exists():candidates=list(Path('/kaggle/input').rglob(filename))
    candidates.append(PRIV/filename)
    for path in candidates:
        if path.is_file() and path.stat().st_size==size and git_blob(path)==blob:
            return path,{'sha256':sha256(path),'bytes':size,'origin':'verified_cache'}
    dst=PRIV/filename; partial=PRIV/(filename+'.part')
    url='https://raw.githubusercontent.com/HarmJ0y/PhishingSpamDataSet/'+D2_REV+'/'+urllib.parse.quote(rel,safe='/')
    try:
        with urllib.request.urlopen(url,timeout=90) as response, partial.open('wb') as f:
            count=0
            while True:
                b=response.read(1<<20)
                if not b:break
                count+=len(b)
                if count>size:raise ValueError('download exceeds pinned size')
                f.write(b)
        if partial.stat().st_size!=size or git_blob(partial)!=blob:raise ValueError('Pinned Git blob mismatch: '+name)
        partial.replace(dst)
    finally:
        if partial.exists():partial.unlink()
    return dst,{'sha256':sha256(dst),'bytes':size,'origin':'pinned_download'}

def get(row,*keys):
    normalized={re.sub('[^a-z0-9]','',str(k).lower()):v for k,v in row.items()}
    for k in keys:
        x=normalized.get(re.sub('[^a-z0-9]','',k.lower()))
        if x is not None:return x
    return None

def load(path):
    with path.open(encoding='utf-8-sig') as f:d=json.load(f)
    if isinstance(d,dict):
        d=next((d[k] for k in ('emails','data','records','items','dataset','results') if isinstance(d.get(k),list)),None)
    if not isinstance(d,list) or not all(isinstance(x,dict) for x in d):raise ValueError('Invalid JSON list')
    return d

def ident(x):
    if x is None or isinstance(x,(dict,list,bool)):return None
    s=str(x).strip()
    if not s or s.casefold() in ('none','null','nan'):return None
    # Source IDs may have numeric JSON and numeric-looking strings.
    if re.fullmatch(r'\d+\.0',s):s=s[:-2]
    return s

def label(r):
    x=get(r,'Type')
    return {'Phishing':1,'Valid':0}.get(x)

def txt(r):return str(get(r,'Subject') or '')+'\n'+str(get(r,'Body') or '')
def norm(s):return ' '.join(s.casefold().split())
def digest_text(s):return hashlib.sha256(norm(s).encode('utf-8')).hexdigest()
def group_index(rows):
    out={}
    for r in rows:
        key=ident(get(r,'No.'))
        if key is None or key in out:raise ValueError('Missing/duplicate original parent ID')
        out[key]=r
    return out

def threshold_at_fpr(y,s,target=0.05):
    # Higher score => phishing. Strict > score_max rejects all; deterministic selection only on val.
    neg=np.asarray(s)[np.asarray(y)==0]
    if len(neg)<20:raise ValueError('Insufficient validation legitimate controls')
    candidates=np.unique(np.r_[np.nextafter(max(s),np.inf),np.asarray(s)])
    best=max(s)+1.0
    for t in candidates:
        fpr=float(np.mean(neg>=t))
        if fpr<=target:best=float(t);break
    return best

def score_metrics(y,s,t):
    y=np.asarray(y,dtype=int);s=np.asarray(s,dtype=float)
    if len(y)==0 or not np.all(np.isfinite(s)):raise ValueError('Invalid/nonfinite predictions')
    pred=(s>=t).astype(int);cm=confusion_matrix(y,pred,labels=[0,1]);tn,fp,fn,tp=cm.ravel()
    bins=np.linspace(0,1,11);ece=0.0
    for b in range(10):
        mask=(s>=bins[b]) & ((s<bins[b+1]) if b<9 else (s<=bins[b+1]))
        if mask.any():ece+=mask.mean()*abs(float(y[mask].mean())-float(s[mask].mean()))
    return {'n':int(len(y)),'positive':int(y.sum()),'legitimate':int(len(y)-y.sum()),
        'precision':float(precision_score(y,pred,zero_division=0)),
        'recall':float(recall_score(y,pred,zero_division=0)),
        'f1':float(f1_score(y,pred,zero_division=0)),
        'fpr':float(fp/(fp+tn)) if fp+tn else None,
        'tp':int(tp),'tn':int(tn),'fp':int(fp),'fn':int(fn),
        'roc_auc':float(roc_auc_score(y,s)) if len(set(y))==2 else None,
        'average_precision':float(average_precision_score(y,s)) if len(set(y))==2 else None,
        'brier':float(brier_score_loss(y,s)), 'ece_10_bins':float(ece)}

def smoke():
    assert ident(3.0)=='3'
    assert norm(' A  B ')=='a b'
    assert label({'Type':'Spam'}) is None
    assert label({'Type':'Phishing Simulation'}) is None
    assert label({'Type':'Valid'})==0
    assert threshold_at_fpr([0]*30+[1]*20,np.linspace(0,1,50),0.05)>0
    m=score_metrics([0,0,1,1],[.1,.8,.7,.9],.5)
    assert m['fp']==1 and m['tp']==2
    return 'PASS'

start=time.monotonic(); test=smoke(); paths={}; sources={}
for name in SOURCE:
    paths[name],sources[name]=acquire(name)
raw={k:load(p) for k,p in paths.items()}
orig=group_index(raw['original'])
assert len(orig)==3075, 'Source revision/row count changed'
counts=Counter(str(get(r,'Type')) for r in orig.values())
assert counts=={'Phishing':1090,'Spam':1000,'Valid':950,'Phishing Simulation':35}, 'Original class taxonomy changed'
valid={k:r for k,r in orig.items() if label(r) is not None and txt(r).strip()}
# Remove normalized-text label disagreements and identical text aliases across different parents.
text_to_groups=defaultdict(set);text_to_classes=defaultdict(set)
for k,r in valid.items():
    h=digest_text(txt(r));text_to_groups[h].add(k);text_to_classes[h].add(label(r))
excluded_keys={k for h,ids in text_to_groups.items() if len(ids)>1 or len(text_to_classes[h])>1 for k in ids}
valid={k:r for k,r in valid.items() if k not in excluded_keys}
assert len(valid)>300 and len(set(label(r) for r in valid.values()))==2

def group_split(keys):
    y=[label(valid[k]) for k in keys]
    train,tmp=train_test_split(keys,test_size=.3,random_state=SEED,stratify=y)
    val,test=train_test_split(tmp,test_size=.5,random_state=SEED,stratify=[label(valid[k]) for k in tmp])
    return train,val,test

keys=sorted(valid)
train_ids,val_ids,test_ids=group_split(keys)
assert not(set(train_ids)&set(val_ids) or set(train_ids)&set(test_ids) or set(val_ids)&set(test_ids))
# Remove samples in validation/test whose text also occurs in training; crossing groups already excluded above.
def dataset(ids):return [txt(valid[k]) for k in ids],np.array([label(valid[k]) for k in ids],dtype=int)
X_train,y_train=dataset(train_ids);X_val,y_val=dataset(val_ids);X_test,y_test=dataset(test_ids)
assert len(set(y_train))==len(set(y_val))==len(set(y_test))==2
# Require sufficient legitimate controls (strict gate, no cherry-picking split).
assert sum(y_val==0)>=20 and sum(y_test==0)>=20

# Rewrite files contain same No. identity; parent test only, never train on test parent rewrites.
rewrites={}; coverage={}
for provider in ('gpt4o','deepseek','randomapi'):
    idx=group_index(raw[provider]); assert len(idx)==3075
    selected=[]; suspicious=0
    for k in test_ids:
        r=idx.get(k)
        if r is None or not txt(r).strip():continue
        # Inconsistent labeled source type is excluded rather than silently inherited.
        source_type=get(r,'Type')
        if source_type is not None and source_type!=get(orig[k],'Type'):
            suspicious+=1;continue
        selected.append((k,txt(r),label(valid[k])))
    rewrites[provider]=selected
    coverage[provider]={'usable_pairs':len(selected),'type_conflicts_excluded':suspicious}
    assert len(selected)>100

# Leakage checks: block rewritten test text if it is identical to original training text.
train_norm={digest_text(x) for x in X_train}
for provider,rows in rewrites.items():
    overlap=sum(digest_text(x) in train_norm for _,x,_ in rows)
    coverage[provider]['identical_to_training_text']=overlap
    if overlap:rewrites[provider]=[(k,x,y) for k,x,y in rows if digest_text(x) not in train_norm]

vec=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_df=.98,sublinear_tf=True,max_features=120000,strip_accents='unicode')
train_mat=vec.fit_transform(X_train); val_mat=vec.transform(X_val); test_mat=vec.transform(X_test)
models={
 'tfidf_logistic':LogisticRegression(max_iter=1500,class_weight='balanced',random_state=SEED),
 'tfidf_linear_svm_calibrated':CalibratedClassifierCV(LinearSVC(class_weight='balanced',random_state=SEED),method='sigmoid',cv=3)
}
results=[]; paired=[]
for model_name,model in models.items():
    model.fit(train_mat,y_train)
    pval=model.predict_proba(val_mat)[:,1]
    threshold=threshold_at_fpr(y_val,pval,.05)
    ptest=model.predict_proba(test_mat)[:,1]
    results.append({'model':model_name,'partition':'heldout_original','threshold_validation_5pct_fpr':threshold,**score_metrics(y_test,ptest,threshold)})
    baseline={k:float(score) for k,score in zip(test_ids,ptest)}
    for provider,rows in rewrites.items():
        ids=[k for k,_,_ in rows]; x=[x for _,x,_ in rows]; y=np.asarray([y for _,_,y in rows],dtype=int)
        scores=model.predict_proba(vec.transform(x))[:,1]
        results.append({'model':model_name,'partition':'heldout_rewrite_'+provider,'threshold_validation_5pct_fpr':threshold,**score_metrics(y,scores,threshold)})
        for cla,name in ((0,'legitimate'),(1,'phishing')):
            mask=(y==cla)
            if not mask.any():continue
            delta=[float(z-baseline[k]) for (k,_,yy),z in zip(rows,scores) if yy==cla]
            orig_wrong=[baseline[k]>=threshold for k,_,yy in rows if yy==cla]
            rewrite_wrong=[z>=threshold for (_,_,yy),z in zip(rows,scores) if yy==cla]
            if cla==1:
                orig_wrong=[not x for x in orig_wrong];rewrite_wrong=[not x for x in rewrite_wrong]
            paired.append({'model':model_name,'provider':provider,'target_class':name,'n':int(mask.sum()),
                'original_error_rate_on_same_parent_subset':float(np.mean(orig_wrong)),
                'rewrite_error_rate':float(np.mean(rewrite_wrong)),
                'rewrite_minus_original_error_rate':float(np.mean(rewrite_wrong)-np.mean(orig_wrong)),
                'mean_rewrite_minus_original_phishing_score':float(np.mean(delta))})

# Save aggregates only. No raw text, row-level scores, sender, URL or IDs in output.
summary={'release':RELEASE,'utc':datetime.now(timezone.utc).isoformat(),'status':'EXECUTED_SOURCE_ASSIGNED_LABEL_BENCHMARK',
 'runtime_seconds':round(time.monotonic()-start,2),'python':sys.version.split()[0],
 'platform':platform.platform(),'self_test':test,'source_integrity':sources,
 'source_class_counts':dict(counts),'excluded_ambiguous_or_duplicate_parent_ids':len(excluded_keys),
 'split_counts':{'train':len(train_ids),'validation':len(val_ids),'test':len(test_ids)},
 'split_label_counts':{z:dict(Counter(map(int,y))) for z,y in [('train',y_train),('val',y_val),('test',y_test)]},
 'source_rewrite_coverage':coverage,'models':[x for x in models],
 'limitations':['D2 labels are source-assigned, not independent gold','GPT-4o/DeepSeek outputs may be model-rewritten but intent not individually human verified',
 'This is matched-parent robustness, not independent unseen-corpus evaluation','RandomAPI is not a verified single-generator identifier',
 'No cross-language D4 comparisons here','Spam excluded from binary target','Source rights/privacy and publication reuse require separate assessment']}

def write_csv(name,rows):
    with (OUT/name).open('w',encoding='utf-8',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
(OUT/'summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
write_csv('metrics.csv',results);write_csv('paired_effects.csv',paired)
(OUT/'README.txt').write_text('N05 E01 safe aggregate outputs. Source-assigned labels, not human gold. No raw email included.\n',encoding='utf-8')
summary['output_sha256']={p.name:sha256(p) for p in sorted(OUT.iterdir()) if p.is_file()}
(OUT/'run_manifest.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print('N05 completed; SAFE AGGREGATES:',OUT)
print('Split counts:',summary['split_counts'],'Excluded duplicated parents:',len(excluded_keys))
for row in paired:print(row['model'],row['provider'],row['target_class'],'n=',row['n'],'error delta=',round(row['rewrite_minus_original_error_rate'],4))


## Interpret safely

`paired_effects.csv` compares original vs rewritten emails within **the same held-out parent IDs** at thresholds frozen on the original validation partition. Negative values do not imply an improvement caused by AI in general; they describe this particular labeled dataset. Review class `Valid` is an inherited **source** annotation, not independently verified benign intent. `RandomAPI` is a mixed pipeline. No external D4 test is run in this notebook.

To share: download only `/kaggle/working/phishing_n05_safe/` metadata and tables. Never export `/kaggle/temp/phishing_n05_private/` or raw messages to a public repository. Commit a reviewed privacy-safe results ZIP and this output-stripped notebook to GitHub.
